# Étape 12 — Le test décisif : le LLM aide-t-il quand ses flèches sont bien réparties ?

L'étape 11 a montré que l'utilité d'un graphe pour le KT dépend surtout de sa **couverture** : il faut que chaque exercice ait des prérequis.
Les graphes LLM « prudents » échouaient parce que leurs flèches se concentraient sur quelques notions de base.

Ici, on corrige ce défaut : **pour chaque exercice B, on garde ses k meilleurs prérequis A selon le LLM** (k = 1, 2, 3, 5).
La couverture devient alors presque complète, quel que soit k.

**Contrôle :** pour chaque exercice B, k prérequis tirés **au hasard parmi les exercices du même thème**. Même couverture, même nombre de flèches, mais sans le LLM.

- **LLM top-k > thème top-k** : le LLM apporte une vraie connaissance, une fois ses flèches bien réparties.
- **LLM top-k ≈ thème top-k** : la structure par thème suffit, le choix du LLM n'ajoute rien.

**Hypothèse principale, fixée avant l'expérience :** LLM top-k contre thème top-k, pour k = 1, 2, 3, 5, dans les deux régimes (8 tests, correction de Holm sur ces 8 tests seulement).

**GPU T4 recommandé.** Environ 80 entraînements, 1 h 30 à 2 h. Reprise automatique en cas de déconnexion. Aucun appel au LLM : on relit ses scores.

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        # print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

## Graphes LLM et catégories de flèches (repris de l'étape 10)

In [ ]:
# Cellule C1 — Paires candidates et couverture du graphe expert
# Périmètre : même thème (topic) par défaut ; même domaine (area) si le thème couvre moins de 70 % des flèches expertes
exos['texte'] = exos['name'].str.replace('_', ' ').str.strip()
def paires_par(colonne):
    groupes = exos.dropna(subset=[colonne]).groupby(colonne)['exo_id'].apply(list)
    return [(a, b) for ids in groupes for a in ids for b in ids if a != b]
couverture = {}
for colonne in ['topic', 'area']:
    c = paires_par(colonne)
    cs = set(c)
    couverture[colonne] = (c, sum(1 for e in aretes_expert if tuple(e) in cs) / len(aretes_expert))
    print(f'Périmètre « {colonne} » : {len(c):,} paires | flèches expertes couvertes : {couverture[colonne][1]:.1%}')
PERIMETRE = 'topic' if couverture['topic'][1] >= 0.70 or len(couverture['area'][0]) > 120_000 else 'area'
candidats, rappel_max = couverture[PERIMETRE]
print(f'\n=> Périmètre retenu : « {PERIMETRE} » — {len(candidats):,} paires, rappel maximal {rappel_max:.1%}')
print('Exemples :', [(exos.texte[a], exos.texte[b]) for a, b in candidats[:3]])

In [ ]:
# Cellule C2 — Graphes LLM : du plus prudent au plus généreux, et leur qualité face au graphe expert
import gc
if 'llm' in globals():
    del llm; gc.collect(); torch.cuda.empty_cache()      # libère le GPU pour les entraînements

scores = pd.read_csv(os.path.join(RESULTATS, 'junyi_llm_scores.csv'))
scores[['A', 'B']] = scores[['A', 'B']].astype(int)
scores = scores.sort_values('p_oui', ascending=False).reset_index(drop=True)
vrai = set(map(tuple, aretes_expert))
scores['expert'] = [(a, b) in vrai for a, b in zip(scores.A, scores.B)]
print(f"AUC du LLM sur les paires candidates : {roc_auc_score(scores.expert, scores.p_oui):.3f}")

E_star = len(aretes_expert)
GRAPHES_LLM = {}
for lib, k in [('LLM_x0.5', 0.5), ('LLM_x1', 1), ('LLM_x2', 2), ('LLM_x4', 4), ('LLM_x8', 8)]:
    GRAPHES_LLM[lib] = scores.head(int(k * E_star))                       # les k × |E*| paires les plus probables
GRAPHES_LLM['LLM_seuil0.5'] = scores[scores.p_oui >= 0.5]                  # décision « oui/non » brute du LLM

qualite = []
for lib, s in GRAPHES_LLM.items():
    E = list(zip(s.A, s.B))
    A = vers_matrice(E)
    tp = int(s.expert.sum())
    qualite.append({'graphe': lib, 'fleches': len(E), 'seuil_equivalent': round(float(s.p_oui.min()), 3) if len(s) else None,
                    'precision': round(tp / max(1, len(E)), 3), 'rappel': round(tp / E_star, 3),
                    'F1': f1_aretes(A), 'acyclique': nx.is_directed_acyclic_graph(nx.DiGraph(E)) if E else True})
qualite = pd.DataFrame(qualite)
qualite.to_csv(os.path.join(RESULTATS, 'junyi_llm_qualite.csv'), index=False)
print(qualite.to_string(index=False))

In [ ]:
# Cellule C3 — Classement des flèches LLM par rapport au graphe expert
G_exp = nx.DiGraph(); G_exp.add_nodes_from(range(N)); G_exp.add_edges_from(map(tuple, aretes_expert))
desc = {n: nx.descendants(G_exp, n) for n in G_exp}
comp = {}
for i, c in enumerate(nx.weakly_connected_components(G_exp)):
    for n in c:
        comp[n] = i
E_dir = set(map(tuple, aretes_expert))

def categorie(a, b):
    if (a, b) in E_dir:      return 'direct'
    if b in desc[a]:         return 'indirect'
    if (b, a) in E_dir:      return 'inverse'
    if a in desc[b]:         return 'indirect_inverse'
    if comp[a] == comp[b]:   return 'meme_composante'
    return 'sans_rapport'

ORDRE = ['direct', 'indirect', 'inverse', 'indirect_inverse', 'meme_composante', 'sans_rapport']
repart = {}
for lib, s in GRAPHES_LLM.items():
    c = pd.Series([categorie(a, b) for a, b in zip(s.A, s.B)]).value_counts(normalize=True)
    repart[lib] = c.reindex(ORDRE, fill_value=0)
repart['hasard (toutes candidates)'] = pd.Series([categorie(a, b) for a, b in candidats]).value_counts(normalize=True).reindex(ORDRE, fill_value=0)
repart = (100 * pd.DataFrame(repart).T).round(1)
repart['precision_elargie_%'] = repart['direct'] + repart['indirect']
repart.to_csv(os.path.join(RESULTATS, 'rq1_categories_fleches_llm.csv'))
print('Répartition des flèches (%) :')
print(repart.to_string())

## D1 — Graphes top-k par exercice

Le LLM est **sur-confiant** : beaucoup de paires ont une probabilité presque égale à 1.
Pour départager ces ex æquo sans favoriser un ordre arbitraire du fichier, on les classe au hasard (graine fixée). La cellule indique combien de flèches retenues sont concernées.

In [ ]:
# Cellule D1 — LLM top-k et thème top-k par exercice, avec leurs mesures de qualité et de couverture
K_LISTE = [1, 2, 3, 5]
GRAINES = [0, 1, 2, 3, 4]

sc = pd.read_csv(os.path.join(RESULTATS, 'junyi_llm_scores.csv'))
sc[['A', 'B']] = sc[['A', 'B']].astype(int)
sc['alea'] = np.random.default_rng(123).random(len(sc))          # départage des ex æquo
sc = sc.sort_values(['B', 'p_oui', 'alea'], ascending=[True, False, True])

def llm_topk(k):
    sel = sc.groupby('B', sort=False).head(k)
    return list(zip(sel.A, sel.B)), float((sel.p_oui >= 0.999).mean())

sources_par_cible = {}
for a, b in candidats:
    sources_par_cible.setdefault(b, []).append(a)

def theme_topk(k, graine):
    rng = random.Random(70_000 + graine)
    return [(a, b) for b, srcs in sorted(sources_par_cible.items()) for a in rng.sample(srcs, min(k, len(srcs)))]

# Premières tentatives du test, pour la couverture
_, Qt, _, Ft = donnees['test']
premieres = pd.Series(np.concatenate([q[f] for q, f in zip(Qt, Ft)])).value_counts()
poids = premieres.reindex(range(N), fill_value=0).values

def mesures(E):
    E = list(set(map(tuple, E)))
    a_prereq = np.zeros(N, bool)
    for _, b in E:
        a_prereq[b] = True
    tp = sum(1 for e in E if e in E_dir)
    cats = pd.Series([categorie(a, b) for a, b in E]).value_counts(normalize=True)
    return {'fleches': len(E), 'precision': tp / max(1, len(E)), 'rappel': tp / len(E_dir),
            'precision_elargie_%': 100 * (cats.get('direct', 0) + cats.get('indirect', 0)),
            'inverse_%': 100 * (cats.get('inverse', 0) + cats.get('indirect_inverse', 0)),
            'couverture_1res_tentatives_%': 100 * (poids * a_prereq).sum() / poids.sum()}

GRAPHES_TOPK, lignes = {}, []
for k in K_LISTE:
    E, ex_aequo = llm_topk(k)
    GRAPHES_TOPK[f'LLM_k{k}'] = {g: E for g in GRAINES}
    lignes.append({'graphe': f'LLM_k{k}', **mesures(E), 'part_ex_aequo_p>=0.999_%': 100 * ex_aequo})
    GRAPHES_TOPK[f'theme_k{k}'] = {g: theme_topk(k, g) for g in GRAINES}
    m = pd.DataFrame([mesures(GRAPHES_TOPK[f'theme_k{k}'][g]) for g in GRAINES]).mean().to_dict()
    lignes.append({'graphe': f'theme_k{k}', **m, 'part_ex_aequo_p>=0.999_%': None})
lignes.append({'graphe': 'expert', **mesures(aretes_expert), 'part_ex_aequo_p>=0.999_%': None})
qual_topk = pd.DataFrame(lignes).round(3)
qual_topk.to_csv(os.path.join(RESULTATS, 'rq1_qualite_topk.csv'), index=False)
print(qual_topk.to_string(index=False))

In [ ]:
# Cellule D2 — PrereqKT avec les graphes top-k (5 % puis 1 %, 5 graines ; reprise automatique)
chemin_topk = os.path.join(RESULTATS, 'rq2_topk.csv')
res_topk = pd.read_csv(chemin_topk).to_dict('records') if os.path.exists(chemin_topk) else []
fait = {(x['graphe'], float(x['fraction']), int(x['graine'])) for x in res_topk}
t0_ = time.time()
for frac in [0.05, 0.01]:
    for k in K_LISTE:
        for famille in ['LLM', 'theme']:
            lib = f'{famille}_k{k}'
            for g_ in GRAINES:
                if (lib, frac, g_) in fait:
                    continue
                res = entrainer('PrereqKT', vers_matrice(GRAPHES_TOPK[lib][g_]), fraction=frac, graine=g_)
                res_topk.append({'modele': 'PrereqKT', 'graphe': lib, 'k': k, 'fraction': frac, 'graine': g_, **res})
                pd.DataFrame(res_topk).to_csv(chemin_topk, index=False)
                print(f'{lib:9s} | {frac:.0%} | graine {g_} -> AUC 1res {res["AUC_1re_test"]} ({(time.time()-t0_)/60:.0f} min)')
print('Entraînements terminés.')

In [ ]:
# Cellule D3 — Bénéfice conservé et tests de l'hypothèse principale (Holm sur 8 tests)
col = 'AUC_1re_test'
ref = pd.read_csv(os.path.join(RESULTATS, 'phase2_confirmation.csv'))
ref = ref[ref.modele == 'PrereqKT'][['graphe', 'fraction', 'graine', col]]
tk = pd.DataFrame(res_topk)[['graphe', 'fraction', 'graine', col]]
tout = pd.concat([ref, tk], ignore_index=True)

lignes = []
for frac, s in tout.groupby('fraction'):
    if frac not in (0.05, 0.01):
        continue
    piv = s.pivot_table(index='graine', columns='graphe', values=col)
    gain = piv['expert'] - piv['aucun']
    for g in piv.columns:
        lignes.append({'fraction': frac, 'graphe': g, 'AUC_1re': round(piv[g].mean(), 4), 'sd': round(piv[g].std(), 4),
                       'benefice_conserve_%': round(100 * ((piv[g] - piv['aucun']) / gain).mean(), 1)})
benef_topk = pd.DataFrame(lignes).sort_values(['fraction', 'benefice_conserve_%'], ascending=[False, False])
benef_topk.to_csv(os.path.join(RESULTATS, 'rq2_topk_benefice.csv'), index=False)

def holm(p):
    p = pd.Series(p); m = p.notna().sum(); o = p.sort_values()
    return (o * (m - np.arange(len(o)))).cummax().clip(upper=1).reindex(p.index)

# Hypothèse principale : LLM_k contre theme_k (appariés par graine)
lignes, pool = [], {k: [] for k in K_LISTE}
for frac in [0.05, 0.01]:
    piv = tout[tout.fraction == frac].pivot_table(index='graine', columns='graphe', values=col)
    for k in K_LISTE:
        a, b = f'LLM_k{k}', f'theme_k{k}'
        if a not in piv or b not in piv:
            continue
        d = (piv[a] - piv[b]).dropna(); pool[k] += list(d)
        lignes.append({'fraction': frac, 'comparaison': f'{a} − {b}', 'n': len(d), 'ecart_moyen': round(d.mean(), 4),
                       'ecart_sd': round(d.std(), 4), 'LLM_gagne': f'{int((d > 0).sum())}/{len(d)}',
                       'p_ttest': stats.ttest_rel(piv[a].loc[d.index], piv[b].loc[d.index]).pvalue})
principal = pd.DataFrame(lignes)
principal['p_holm'] = holm(principal['p_ttest'])
principal['significatif_holm_5%'] = principal['p_holm'] < 0.05
principal[['p_ttest', 'p_holm']] = principal[['p_ttest', 'p_holm']].round(4)
principal.to_csv(os.path.join(RESULTATS, 'rq2_topk_tests_principaux.csv'), index=False)

# Regroupement des deux régimes (10 paires par k)
regroupe = pd.DataFrame([{'k': k, 'n_paires': len(v), 'ecart_moyen': round(np.mean(v), 4),
                          'LLM_gagne': f'{int(np.sum(np.array(v) > 0))}/{len(v)}',
                          'p_signe': round(stats.binomtest(int(np.sum(np.array(v) > 0)), len(v)).pvalue, 4),
                          'p_ttest_1echantillon': round(stats.ttest_1samp(v, 0).pvalue, 4)}
                         for k, v in pool.items() if len(v) > 1])

print(qual_topk.to_string(index=False))
for frac in [0.05, 0.01]:
    print(f'\n===== Bénéfice conservé | {frac:.0%} des élèves =====')
    print(benef_topk[benef_topk.fraction == frac].drop(columns='fraction').to_string(index=False))
print('\n===== Hypothèse principale : LLM top-k − thème top-k (correction de Holm sur ces 8 tests) =====')
print(principal.to_string(index=False))
print('\n===== Deux régimes regroupés =====')
print(regroupe.to_string(index=False))

In [ ]:
# Cellule D4 — Figure : bénéfice conservé selon k, LLM contre thème aléatoire
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, frac in zip(axes, [0.05, 0.01]):
    b = benef_topk[benef_topk.fraction == frac].set_index('graphe')['benefice_conserve_%']
    ax.plot(K_LISTE, [b.get(f'LLM_k{k}', np.nan) for k in K_LISTE], 'o-', label='LLM : k meilleurs prérequis par exercice')
    ax.plot(K_LISTE, [b.get(f'theme_k{k}', np.nan) for k in K_LISTE], 's--', label='thème aléatoire : k prérequis par exercice')
    ax.axhline(100, ls=':', c='black', lw=1, label='expert')
    ax.axhline(b.get('aleatoire', np.nan), ls='--', c='grey', lw=1, label='aléatoire global')
    ax.set_xticks(K_LISTE); ax.set_xlabel('k (prérequis par exercice)'); ax.set_title(f'{frac:.0%} des élèves')
axes[0].set_ylabel('Bénéfice conservé (%)'); axes[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(os.path.join(RESULTATS, 'rq2_topk.png'), dpi=200)
plt.show()